# Data Filtration — BoM NSW Rainfall Network (1858–2024)

This notebook implements the quality-control procedure described in the manuscript:

> To investigate heavy rainfall with observations, we applied a data quality control procedure to identify erroneous
> values that would impact the subsequent analysis. We removed all rainfall observations with quality codes
> "N," "W," "S," "I," and "X" for the period 1858–2019. We retained non-quality-controlled rainfall data after 2019
> due to a likely delay in the quality control process for recent observations, as quality codes are not yet
> available (personal communication, BoM 2025). We removed all tagged accumulated rainfall values for the
> entire period 1858–2024 (as shown in the supplementary file Fig. S1).

**Input:** one raw parquet file per station (5,429 stations), produced from the original BoM `.txt` exports
(see the station-file conversion cells in the original notebook — that step is unchanged and lives upstream of this one).

**Output:** cleaned per-station parquet files + a network-wide summary of how many values were removed and why.


## Step 0 — Paths and QC parameters

*Update `RAW_DATA_DIR` / `CLEAN_DATA_DIR` to match your machine.*

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Paths (edit these) --------------------------------------------------
RAW_DATA_DIR   = Path("/Users/")
CLEAN_DATA_DIR = Path("/Users/")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---- QC parameters (must match the manuscript methods paragraph) --------
QC_CUTOFF_YEAR     = 2019                       # quality-code filter applies for Year <= this
BAD_QUALITY_CODES  = ["N", "W", "S", "I", "X"]  # codes removed for 1858-2019

RAINFALL_COL_RAW = "Precipitation in the 24 hours before 9am (local time) in mm"
QUALITY_COL      = "Quality of precipitation value"
ACCUM_DAYS_COL    = "Accumulated number of days over which the precipitation was measured"
RAIN_DAYS_IN_ACCUM_COL = "Number of days of rain within the days of accumulation"

NUMERIC_COLS = [
    "Station Number", "Year", "Month", "Day", "Rainfall",
    RAIN_DAYS_IN_ACCUM_COL, ACCUM_DAYS_COL,
]

## Step 1 — Standardise columns

Rename the precipitation column to `Rainfall` and coerce numeric fields (unchanged from the original pipeline).

In [ ]:
def standardise(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={RAINFALL_COL_RAW: "Rainfall"})
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

## Step 2 — Quality-code filtering (1858–2019 only)

> We removed all rainfall observations with quality codes "N," "W," "S," "I," and "X" for the period 1858–2019.
> We retained non-quality-controlled rainfall data after 2019 ... as quality codes are not yet available.



In [ ]:
def apply_quality_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Set Rainfall to NaN where the quality code is unreliable, for Year <= QC_CUTOFF_YEAR only.
    Years after the cutoff are left untouched regardless of quality code."""
    df = df.copy()
    is_historic     = df["Year"] <= QC_CUTOFF_YEAR
    has_bad_quality = df[QUALITY_COL].isin(BAD_QUALITY_CODES)

    to_remove = is_historic & has_bad_quality
    df.loc[to_remove, "Rainfall"] = np.nan

    return df, int(to_remove.sum())

## Step 3 — Remove tagged-accumulated values (entire period, 1858–2024)

> We removed all tagged accumulated rainfall values for the entire period 1858–2024 (Fig. S1).

A record is "accumulated" when it represents rainfall summed over more than one day, flagged via
`Accumulated number of days over which the precipitation was measured` > 1. This applies for the full
period, independent of quality code — unlike the original notebook, which only flagged this for
quality-`'Y'` rows.

In [ ]:
def remove_accumulated(df: pd.DataFrame) -> pd.DataFrame:
    """Set Rainfall to NaN wherever the record is tagged as an accumulation of >1 day, for the full period."""
    df = df.copy()
    accumulated_flag = df[ACCUM_DAYS_COL] > 1
    df.loc[accumulated_flag, "Rainfall"] = np.nan
    return df, int(accumulated_flag.sum())

## Step 4 — Run the pipeline over every station file, save cleaned output + removal log

In [ ]:
summary_rows = []

for file_path in sorted(RAW_DATA_DIR.glob("*.parquet")):
    station = file_path.stem
    df = pd.read_parquet(file_path)
    df = standardise(df)

    df, n_quality_removed = apply_quality_filter(df)
    df, n_accum_removed   = remove_accumulated(df)

    df.to_parquet(CLEAN_DATA_DIR / file_path.name, index=False)

    summary_rows.append({
        "Station": station,
        "n_records": len(df),
        "n_removed_quality_code": n_quality_removed,
        "n_removed_accumulated": n_accum_removed,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(CLEAN_DATA_DIR.parent / "qc_filtration_summary.csv", index=False)
summary_df.head()

## Step 5 — Network-wide totals (sanity check against manuscript numbers)

In [ ]:
print(f"Stations processed:             {len(summary_df)}")
print(f"Total values removed (quality): {summary_df['n_removed_quality_code'].sum():,}")
print(f"Total values removed (accum.):  {summary_df['n_removed_accumulated'].sum():,}")

## Step 6 (optional) — Figure S1: removed values by reason

In [ ]:
import matplotlib.pyplot as plt

reasons = ["Bad quality code\n(N/W/S/I/X, \u22642019)", "Tagged accumulated\n(1858\u20132024)"]
counts  = [summary_df["n_removed_quality_code"].sum(), summary_df["n_removed_accumulated"].sum()]

plt.figure(figsize=(6, 4))
plt.bar(reasons, counts, color=["#4C72B0", "#DD8452"])
plt.ylabel("Number of rainfall values removed")
plt.title("Rainfall values removed during quality control")
plt.tight_layout()
# plt.savefig("../analysis/figures/fig_s1_qc_removed.png", dpi=300)
plt.show()